# March Madness 2026 — Championship Predictor

Full tournament recap (actual vs model) and a fresh predictive model for the championship game: **UConn (#2) vs Michigan (#1)**, April 6, 2026.

This notebook is self-contained. Just upload to Colab and Runtime > Run all. Expects data at `/content/drive/MyDrive/March Madness 2026/data/` (already there from prior runs).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATA_DIR = '/content/drive/MyDrive/March Madness 2026/data'
assert os.path.isdir(DATA_DIR), f"Data folder not found: {DATA_DIR}"
print("Data files:", len(os.listdir(DATA_DIR)))

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (GradientBoostingClassifier, RandomForestClassifier,
                              StackingClassifier, HistGradientBoostingClassifier)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss

np.random.seed(42)

## 2. Embedded tournament state (what actually happened)

In [ ]:
STATE = json.loads(r'''{
  "last_updated": "2026-04-05",
  "completed_rounds": ["Round of 64", "Round of 32", "Sweet 16", "Elite Eight", "Final Four"],
  "actual_results": {
    "Sweet 16": [
      {"winner": "Duke", "winner_seed": 1, "loser": "St. John's", "loser_seed": 5, "score": "80-75", "model_pick": "Duke", "model_confidence": 83.1, "correct": true},
      {"winner": "UConn", "winner_seed": 2, "loser": "Michigan State", "loser_seed": 3, "score": "67-63", "model_pick": "UConn", "model_confidence": 62.1, "correct": true},
      {"winner": "Iowa", "winner_seed": 9, "loser": "Nebraska", "loser_seed": 4, "score": "77-71", "model_pick": "Nebraska", "model_confidence": 63.0, "correct": false},
      {"winner": "Illinois", "winner_seed": 3, "loser": "Houston", "loser_seed": 2, "score": "65-55", "model_pick": "Houston", "model_confidence": 63.2, "correct": false},
      {"winner": "Purdue", "winner_seed": 2, "loser": "Texas", "loser_seed": 11, "score": "79-77", "model_pick": "Purdue", "model_confidence": 78.0, "correct": true},
      {"winner": "Arizona", "winner_seed": 1, "loser": "Arkansas", "loser_seed": 4, "score": "109-88", "model_pick": "Arizona", "model_confidence": 74.0, "correct": true},
      {"winner": "Michigan", "winner_seed": 1, "loser": "Alabama", "loser_seed": 4, "score": "90-77", "model_pick": "Michigan", "model_confidence": 70.0, "correct": true},
      {"winner": "Tennessee", "winner_seed": 6, "loser": "Iowa State", "loser_seed": 2, "score": "76-62", "model_pick": "Iowa State", "model_confidence": 61.0, "correct": false}
    ],
    "Elite Eight": [
      {"winner": "Arizona", "winner_seed": 1, "loser": "Purdue", "loser_seed": 2, "score": "79-64", "region": "West", "model_regional_pick": "Arizona", "correct": true},
      {"winner": "Illinois", "winner_seed": 3, "loser": "Iowa", "loser_seed": 9, "score": "71-59", "region": "South", "model_regional_pick": "Houston", "correct": false},
      {"winner": "Michigan", "winner_seed": 1, "loser": "Tennessee", "loser_seed": 6, "score": "95-62", "region": "Midwest", "model_regional_pick": "Michigan", "correct": true},
      {"winner": "UConn", "winner_seed": 2, "loser": "Duke", "loser_seed": 1, "score": "73-72", "region": "East", "model_regional_pick": "Duke", "correct": false}
    ],
    "Final Four": [
      {"winner": "UConn", "winner_seed": 2, "loser": "Illinois", "loser_seed": 3, "score": "71-62", "date": "2026-04-04", "notes": "Neither team was a model Final Four pick (model had Duke vs Houston in this semi)."},
      {"winner": "Michigan", "winner_seed": 1, "loser": "Arizona", "loser_seed": 1, "score": "91-73", "date": "2026-04-04", "notes": "Both teams were model Final Four picks. Aday Mara led Michigan with 26 points."}
    ]
  },
  "final_four": {
    "teams": ["Arizona", "Illinois", "Michigan", "UConn"],
    "matchups": [
      {"game": 1, "team1": "UConn", "team1_seed": 2, "team2": "Illinois", "team2_seed": 3, "time": "6:00 PM ET", "date": "2026-04-04"},
      {"game": 2, "team1": "Arizona", "team1_seed": 1, "team2": "Michigan", "team2_seed": 1, "time": "8:30 PM ET", "date": "2026-04-04"}
    ],
    "championship": {"date": "2026-04-06", "time": "8:30 PM ET"},
    "location": "Lucas Oil Stadium, Indianapolis",
    "network": "TBS"
  },
  "model_accuracy": {
    "round_of_64": {"correct": 22, "total": 32, "pct": 68.8},
    "round_of_32": {"correct": 11, "total": 16, "pct": 68.8},
    "sweet_16": {"correct": 5, "total": 8, "pct": 62.5},
    "elite_eight": {"correct": 2, "total": 4, "pct": 50.0},
    "cumulative": {"correct": 40, "total": 60, "pct": 66.7}
  },
  "model_final_four_picks": ["Duke", "Houston", "Arizona", "Michigan"],
  "model_final_four_correct": 2,
  "model_championship_pick": "Duke",
  "championship_pick_alive": false
}
''')
print("Completed rounds:", STATE['completed_rounds'])
print("Championship:", "UConn (#2) vs Michigan (#1) —", STATE['final_four']['championship']['date'])

## 3. Load team data + historical tournament games

In [ ]:
kb = pd.read_csv(f"{DATA_DIR}/KenPom Barttorvik.csv")
resume = pd.read_csv(f"{DATA_DIR}/Resumes.csv")
matchups = pd.read_csv(f"{DATA_DIR}/Tournament Matchups.csv")

print(f"KenPom Barttorvik: {kb.shape}")
print(f"Resumes:           {resume.shape}")
print(f"Tournament games:  {matchups.shape}")
print(f"Years: {kb['YEAR'].min()}–{kb['YEAR'].max()}")

## 4. Feature engineering

In [ ]:
KB_FEATURES = ['KADJ O','KADJ D','KADJ EM','BADJ EM','BADJ O','BADJ D','BARTHAG','KADJ T',
               'WIN%','EFG%','EFG%D','FTR','FTRD','TOV%','TOV%D','OREB%','DREB%',
               '2PT%','2PT%D','3PT%','3PT%D','FT%','2PTR','3PTR','BLK%','AST%','OP AST%',
               'PPPO','PPPD','ELITE SOS','WAB','EXP','TALENT','AVG HGT','EFF HGT']
RESUME_FEATURES = ['ELO','NET RPI','Q1 W','Q2 W','Q1 PLUS Q2 W','Q3 Q4 L','PLUS 500','R SCORE']

def engineer(row):
    f = {}
    f['NET_EFF']       = row.get('KADJ O',0) - row.get('KADJ D',0)
    f['OFF_DEF_RATIO'] = row.get('KADJ O',0) / max(row.get('KADJ D',1),1)
    f['FOUR_FACTORS_O']= 0.4*row.get('EFG%',0) + 0.25*(100-row.get('TOV%',0)) + 0.20*row.get('OREB%',0) + 0.15*row.get('FTR',0)
    f['FOUR_FACTORS_D']= 0.4*(100-row.get('EFG%D',50)) + 0.25*row.get('TOV%D',0) + 0.20*(100-row.get('OP OREB%',50)) + 0.15*(100-row.get('FTRD',0))
    f['SHOOT_VERSATILITY'] = row.get('2PT%',0)*0.5 + row.get('3PT%',0)*0.5
    f['DEF_PRESSURE']  = row.get('BLK%',0) + row.get('TOV%D',0)
    f['BALL_SECURITY'] = 100 - row.get('TOV%',0)
    f['REB_MARGIN']    = row.get('OREB%',0) + row.get('DREB%',0) - 100
    return f

ENGINEERED = ['NET_EFF','OFF_DEF_RATIO','FOUR_FACTORS_O','FOUR_FACTORS_D',
              'SHOOT_VERSATILITY','DEF_PRESSURE','BALL_SECURITY','REB_MARGIN']
ALL_FEATS = KB_FEATURES + RESUME_FEATURES + ENGINEERED

def team_features(team_no, team_name, year):
    feats = {}
    m = kb[(kb['YEAR']==year) & (kb['TEAM NO']==team_no)]
    if len(m)==0:
        m = kb[(kb['YEAR']==year) & (kb['TEAM']==team_name)]
    if len(m) > 0:
        row = m.iloc[0]
        for c in KB_FEATURES:
            if c in row.index: feats[c] = row[c]
        feats['OP OREB%'] = row.get('OP OREB%',0)
        feats.update(engineer({**feats, **row.to_dict()}))
    r = resume[(resume['YEAR']==year) & (resume['TEAM NO']==team_no)]
    if len(r)==0:
        r = resume[(resume['YEAR']==year) & (resume['TEAM']==team_name)]
    if len(r) > 0:
        row = r.iloc[0]
        for c in RESUME_FEATURES:
            if c in row.index: feats[c] = row[c]
    return feats if feats else None

## 5. Build training set from all historical tournament games

In [ ]:
games = []
for yr in sorted(matchups['YEAR'].unique()):
    for rnd in matchups[matchups['YEAR']==yr]['CURRENT ROUND'].unique():
        sub = matchups[(matchups['YEAR']==yr) & (matchups['CURRENT ROUND']==rnd)].sort_values('BY YEAR NO')
        teams = sub.to_dict('records')
        for i in range(0, len(teams)-1, 2):
            t1, t2 = teams[i], teams[i+1]
            if pd.isna(t1.get('SCORE')) or pd.isna(t2.get('SCORE')): continue
            if t1['ROUND'] < t2['ROUND']: w, l = t1, t2
            elif t2['ROUND'] < t1['ROUND']: w, l = t2, t1
            elif t1['SCORE'] > t2['SCORE']: w, l = t1, t2
            else: w, l = t2, t1
            games.append({'YEAR':yr,'W_NO':w['TEAM NO'],'W':w['TEAM'],'W_SEED':w['SEED'],
                          'L_NO':l['TEAM NO'],'L':l['TEAM'],'L_SEED':l['SEED']})
games_df = pd.DataFrame(games)
print(f"Parsed {len(games_df)} historical tournament games")

rows = []
for _, g in games_df.iterrows():
    wf = team_features(g['W_NO'], g['W'], g['YEAR'])
    lf = team_features(g['L_NO'], g['L'], g['YEAR'])
    if not wf or not lf: continue
    r = {'SEED_DIFF': g['W_SEED']-g['L_SEED'],
         'SEED_SUM':  g['W_SEED']+g['L_SEED'],
         'SEED_PRODUCT': g['W_SEED']*g['L_SEED']}
    for feat in ALL_FEATS:
        if feat in wf and feat in lf:
            r[f'{feat}_DIFF'] = wf[feat] - lf[feat]
    rows.append(r)
train_df = pd.DataFrame(rows)
print(f"Training set: {len(train_df)} games, {train_df.shape[1]} features")

## 6. Train stacked ensemble (LR + GB + RF + HGB)

In [ ]:
feature_cols = ['SEED_DIFF','SEED_SUM','SEED_PRODUCT']
feature_cols += [c for c in train_df.columns if c.endswith('_DIFF') and c not in feature_cols]

nan_pct = train_df[feature_cols].isna().mean()
feature_cols = [c for c in feature_cols if nan_pct[c] < 0.3]
train_clean = train_df[feature_cols].fillna(0)

X = train_clean.values
X_flip = -X.copy()
if 'SEED_SUM' in feature_cols:     X_flip[:, feature_cols.index('SEED_SUM')]     = X[:, feature_cols.index('SEED_SUM')]
if 'SEED_PRODUCT' in feature_cols: X_flip[:, feature_cols.index('SEED_PRODUCT')] = X[:, feature_cols.index('SEED_PRODUCT')]

X_full = np.vstack([X, X_flip])
y_full = np.concatenate([np.ones(len(X)), np.zeros(len(X))])
rng = np.random.RandomState(42)
idx = rng.permutation(len(X_full))
X_full, y_full = X_full[idx], y_full[idx]

print(f"Training on {len(X_full)} samples × {len(feature_cols)} features")

base = [
    ('lr',  Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(C=0.5, max_iter=2000))])),
    ('gb',  GradientBoostingClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                       subsample=0.8, min_samples_leaf=10, random_state=42)),
    ('rf',  RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=10, random_state=42, n_jobs=-1)),
    ('hgb', HistGradientBoostingClassifier(max_iter=300, max_depth=5, learning_rate=0.05,
                                           min_samples_leaf=20, random_state=42)),
]
model = StackingClassifier(estimators=base,
                           final_estimator=LogisticRegression(C=1.0, max_iter=1000),
                           cv=3, stack_method='predict_proba', n_jobs=-1)

X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_full, test_size=0.2, random_state=42, stratify=y_full)
model.fit(X_tr, y_tr)
acc = model.score(X_te, y_te)
brier = brier_score_loss(y_te, model.predict_proba(X_te)[:,1])
print(f"Holdout accuracy: {acc:.4f}   Brier: {brier:.4f}")

model.fit(X_full, y_full)

gb = model.estimators_[1]
feat_imp = sorted(zip(feature_cols, gb.feature_importances_), key=lambda x: -x[1])
print("\nTop 15 features:")
for n, i in feat_imp[:15]:
    print(f"  {n:<30} {i:.4f}")

## 7. Full tournament recap — Actual vs Model

In [ ]:
acc_stats = STATE['model_accuracy']
print("="*70)
print(f"{'MODEL ACCURACY BY ROUND (as originally picked before each round)':^70}")
print("="*70)
for r, d in acc_stats.items():
    if r == 'cumulative': continue
    name = r.replace('_',' ').title()
    bar = '█' * int(d['pct']/5)
    print(f"  {name:<15} {d['correct']:>2}/{d['total']:<2}  {d['pct']:>5.1f}%  {bar}")
c = acc_stats['cumulative']
print(f"  {'─'*40}")
print(f"  {'CUMULATIVE':<15} {c['correct']:>2}/{c['total']:<2}  {c['pct']:>5.1f}%")
print()
print(f"Model Final Four picks:     {STATE['model_final_four_picks']}")
print(f"  Correct:                  {STATE['model_final_four_correct']}/4  (Arizona, Michigan hit; Duke, Houston missed)")
print(f"Model championship pick:    {STATE['model_championship_pick']}  (eliminated by UConn in Elite Eight)")

In [ ]:
print("="*70)
print(f"{'ACTUAL RESULTS — Sweet 16 onward':^70}")
print("="*70)
for rnd in ['Sweet 16','Elite Eight','Final Four']:
    print(f"\n  {rnd}:")
    for g in STATE['actual_results'][rnd]:
        w = f"({g['winner_seed']}) {g['winner']}"
        l = f"({g['loser_seed']}) {g['loser']}"
        score = g.get('score','')
        mp = g.get('model_pick') or g.get('model_regional_pick')
        tag = ''
        if mp is not None:
            tag = '  ✓' if mp == g['winner'] else f'  ✗ (model: {mp})'
        print(f"    {w:<24} over {l:<24} {score:<10}{tag}")
print("\n  Championship (TO BE PLAYED):")
print(f"    (2) UConn  vs  (1) Michigan    — Apr 6, 2026, Lucas Oil Stadium")

## 8. Build a fresh championship predictor

Re-train on historical games restricted to **Elite Eight + Final Four + Championship** games only. Late-round games have a different character than R64 upsets, and since only one game remains it's worth asking the model a round-specific question.

In [ ]:
# Re-tag rounds: identify Elite Eight+, Final Four, Championship in matchups
# In Tournament Matchups.csv, CURRENT ROUND values represent the round number
# 64, 32, 16, 8, 4, 2  →  R64, R32, S16, E8, F4, Championship
LATE_ROUNDS = [8, 4, 2]

late_games = []
for yr in sorted(matchups['YEAR'].unique()):
    sub = matchups[(matchups['YEAR']==yr) & (matchups['CURRENT ROUND'].isin(LATE_ROUNDS))]
    for rnd in sub['CURRENT ROUND'].unique():
        rsub = sub[sub['CURRENT ROUND']==rnd].sort_values('BY YEAR NO')
        teams = rsub.to_dict('records')
        for i in range(0, len(teams)-1, 2):
            t1, t2 = teams[i], teams[i+1]
            if pd.isna(t1.get('SCORE')) or pd.isna(t2.get('SCORE')): continue
            if t1['ROUND'] < t2['ROUND']: w, l = t1, t2
            elif t2['ROUND'] < t1['ROUND']: w, l = t2, t1
            elif t1['SCORE'] > t2['SCORE']: w, l = t1, t2
            else: w, l = t2, t1
            late_games.append({'YEAR':yr,'W_NO':w['TEAM NO'],'W':w['TEAM'],'W_SEED':w['SEED'],
                               'L_NO':l['TEAM NO'],'L':l['TEAM'],'L_SEED':l['SEED']})

print(f"Late-round training games (E8/F4/Championship): {len(late_games)}")

late_rows = []
for g in late_games:
    wf = team_features(g['W_NO'], g['W'], g['YEAR'])
    lf = team_features(g['L_NO'], g['L'], g['YEAR'])
    if not wf or not lf: continue
    r = {'SEED_DIFF': g['W_SEED']-g['L_SEED'],
         'SEED_SUM':  g['W_SEED']+g['L_SEED'],
         'SEED_PRODUCT': g['W_SEED']*g['L_SEED']}
    for feat in ALL_FEATS:
        if feat in wf and feat in lf:
            r[f'{feat}_DIFF'] = wf[feat] - lf[feat]
    late_rows.append(r)
late_df = pd.DataFrame(late_rows)
print(f"Late-round feature rows: {len(late_df)}")

In [ ]:
late_feats = [c for c in feature_cols if c in late_df.columns]
Xl = late_df[late_feats].fillna(0).values
Xl_flip = -Xl.copy()
if 'SEED_SUM' in late_feats:     Xl_flip[:, late_feats.index('SEED_SUM')]     = Xl[:, late_feats.index('SEED_SUM')]
if 'SEED_PRODUCT' in late_feats: Xl_flip[:, late_feats.index('SEED_PRODUCT')] = Xl[:, late_feats.index('SEED_PRODUCT')]
Xl_full = np.vstack([Xl, Xl_flip])
yl_full = np.concatenate([np.ones(len(Xl)), np.zeros(len(Xl))])
perm = np.random.RandomState(7).permutation(len(Xl_full))
Xl_full, yl_full = Xl_full[perm], yl_full[perm]

# Smaller dataset → simpler ensemble
late_base = [
    ('lr',  Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(C=0.5, max_iter=2000))])),
    ('gb',  GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                                       subsample=0.8, min_samples_leaf=8, random_state=42)),
    ('hgb', HistGradientBoostingClassifier(max_iter=200, max_depth=4, learning_rate=0.05,
                                           min_samples_leaf=15, random_state=42)),
]
late_model = StackingClassifier(estimators=late_base,
                                final_estimator=LogisticRegression(C=1.0, max_iter=1000),
                                cv=5, n_jobs=-1)
X_tr, X_te, y_tr, y_te = train_test_split(Xl_full, yl_full, test_size=0.2, random_state=42, stratify=yl_full)
late_model.fit(X_tr, y_tr)
late_acc = late_model.score(X_te, y_te)
late_brier = brier_score_loss(y_te, late_model.predict_proba(X_te)[:,1])
print(f"Late-round model  holdout accuracy: {late_acc:.4f}  Brier: {late_brier:.4f}")
late_model.fit(Xl_full, yl_full)

## 9. Predict the championship — UConn (#2) vs Michigan (#1)

In [ ]:
uconn = team_features(None, 'Connecticut', 2026)
mich  = team_features(None, 'Michigan', 2026)
if not uconn: uconn = team_features(None, 'UConn', 2026)
assert uconn and mich, "Couldn't find team features"

def build_row(t1f, t2f, s1, s2, cols):
    r = {'SEED_DIFF': s1-s2, 'SEED_SUM': s1+s2, 'SEED_PRODUCT': s1*s2}
    for feat in ALL_FEATS:
        col = f'{feat}_DIFF'
        if col in cols:
            r[col] = (t1f.get(feat, 0) or 0) - (t2f.get(feat, 0) or 0)
    return np.array([[r.get(c, 0) for c in cols]])

# Full model prediction
row_full = build_row(uconn, mich, 2, 1, feature_cols)
p_uconn_full = model.predict_proba(row_full)[0][1]

# Late-round specialist model prediction
row_late = build_row(uconn, mich, 2, 1, late_feats)
p_uconn_late = late_model.predict_proba(row_late)[0][1]

# Blend (simple average)
p_uconn = 0.5 * p_uconn_full + 0.5 * p_uconn_late

print("="*60)
print(f"{'NCAA CHAMPIONSHIP GAME PREDICTION':^60}")
print(f"{'April 6, 2026 · Lucas Oil Stadium':^60}")
print("="*60)
print(f"\n  Full stacked ensemble:  UConn {p_uconn_full*100:.1f}%  |  Michigan {(1-p_uconn_full)*100:.1f}%")
print(f"  Late-round specialist:  UConn {p_uconn_late*100:.1f}%  |  Michigan {(1-p_uconn_late)*100:.1f}%")
print(f"\n  Blended pick:           UConn {p_uconn*100:.1f}%  |  Michigan {(1-p_uconn)*100:.1f}%")
winner = 'UConn' if p_uconn > 0.5 else 'Michigan'
print(f"\n  >>> PREDICTED WINNER:  {winner}  ({max(p_uconn, 1-p_uconn)*100:.1f}%)")

## 10. Monte Carlo — uncertainty bands via feature perturbation

In [ ]:
# Inject Gaussian noise on the feature diffs to approximate measurement/variance uncertainty
N_SIMS = 10000
NOISE_SCALE = 0.10  # 10% of feature std

stds_full = train_clean.std().values
stds_late = late_df[late_feats].fillna(0).std().values

base_full = row_full.flatten()
base_late = row_late.flatten()

rng = np.random.RandomState(2026)
noise_full = rng.normal(0, NOISE_SCALE * stds_full, size=(N_SIMS, len(feature_cols)))
noise_late = rng.normal(0, NOISE_SCALE * stds_late, size=(N_SIMS, len(late_feats)))

probs_full = model.predict_proba(base_full + noise_full)[:, 1]
probs_late = late_model.predict_proba(base_late + noise_late)[:, 1]
probs_blend = 0.5 * probs_full + 0.5 * probs_late

uconn_wins  = (probs_blend > 0.5).sum()
mich_wins   = N_SIMS - uconn_wins

p5, p50, p95 = np.percentile(probs_blend, [5, 50, 95])

print(f"Monte Carlo ({N_SIMS:,} sims, blended model):")
print(f"  UConn win share:         {uconn_wins/N_SIMS*100:>5.1f}%")
print(f"  Michigan win share:      {mich_wins/N_SIMS*100:>5.1f}%")
print()
print(f"  P(UConn wins) quantiles:")
print(f"    5th  percentile:       {p5*100:.1f}%")
print(f"    median:                {p50*100:.1f}%")
print(f"    95th percentile:       {p95*100:.1f}%")
print()
print(f"  Mean prob UConn:         {probs_blend.mean()*100:.1f}%  (±{probs_blend.std()*100:.1f})")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(probs_blend*100, bins=50, color='#f97316', edgecolor='#0a0e17', alpha=0.85)
ax.axvline(50, color='#64748b', linestyle='--', linewidth=1, label='Coin flip (50%)')
ax.axvline(probs_blend.mean()*100, color='#3b82f6', linewidth=2, label=f'Mean P(UConn) = {probs_blend.mean()*100:.1f}%')
ax.set_xlabel('P(UConn wins) — %')
ax.set_ylabel('Simulation count')
ax.set_title('Championship Predictor — Monte Carlo Distribution (UConn vs Michigan)')
ax.legend()
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

## 11. Score projection (bonus)

In [ ]:
# Tempo-adjusted score projection using KADJ O/D and tempo
def project_score(t1, t2):
    # Expected possessions via harmonic-ish blend of tempos (KenPom style)
    tempo = (t1.get('KADJ T', 70) + t2.get('KADJ T', 70)) / 2
    # PPP for team1 offense vs team2 defense, scaled to 100
    t1_off = t1.get('KADJ O', 110)
    t1_def = t1.get('KADJ D', 100)
    t2_off = t2.get('KADJ O', 110)
    t2_def = t2.get('KADJ D', 100)
    league_avg = 105
    t1_exp = ((t1_off + t2_def - league_avg) / 100) * tempo
    t2_exp = ((t2_off + t1_def - league_avg) / 100) * tempo
    return t1_exp, t2_exp, tempo

uc, mi, tempo = project_score(uconn, mich)
print(f"Projected tempo (possessions): {tempo:.1f}")
print(f"Projected score:  UConn {uc:.1f}  —  Michigan {mi:.1f}")
print(f"Projected margin: {'UConn' if uc>mi else 'Michigan'} by {abs(uc-mi):.1f}")

## Summary

- **Model accuracy to date:** 40/60 (66.7%) through Elite Eight
- **Original champion pick** (Duke) eliminated by UConn in Elite Eight
- **Final Four:** model got 2/4 (Arizona, Michigan correct)
- **Championship:** this notebook's blended prediction for UConn vs Michigan above

Re-upload to Colab and Run all. Nothing runs locally.

## 12. Save outputs to Drive

In [ ]:
OUT_DIR = '/content/drive/MyDrive/March Madness 2026'
os.makedirs(OUT_DIR, exist_ok=True)

# Save prediction JSON
prediction = {
    'game': 'NCAA Championship 2026',
    'date': '2026-04-06',
    'location': 'Lucas Oil Stadium, Indianapolis',
    'teams': {'uconn_seed': 2, 'michigan_seed': 1},
    'models': {
        'full_ensemble':      {'p_uconn': float(p_uconn_full), 'p_michigan': float(1-p_uconn_full)},
        'late_round_specialist': {'p_uconn': float(p_uconn_late), 'p_michigan': float(1-p_uconn_late)},
        'blended':            {'p_uconn': float(p_uconn),      'p_michigan': float(1-p_uconn)},
    },
    'predicted_winner': winner,
    'monte_carlo': {
        'n_sims': int(N_SIMS),
        'uconn_win_share':    float(uconn_wins/N_SIMS),
        'michigan_win_share': float(mich_wins/N_SIMS),
        'p_uconn_mean':       float(probs_blend.mean()),
        'p_uconn_std':        float(probs_blend.std()),
        'p_uconn_p5':         float(p5),
        'p_uconn_p50':        float(p50),
        'p_uconn_p95':        float(p95),
    },
    'score_projection': {
        'tempo': float(tempo),
        'uconn': float(uc),
        'michigan': float(mi),
        'margin': float(abs(uc-mi)),
        'projected_winner': 'UConn' if uc > mi else 'Michigan',
    },
    'model_accuracy_through_elite_eight': STATE['model_accuracy'],
}
pred_path = f'{OUT_DIR}/championship_prediction.json'
with open(pred_path, 'w') as f:
    json.dump(prediction, f, indent=2)
print(f'Saved: {pred_path}')

# Save histogram PNG
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(probs_blend*100, bins=50, color='#f97316', edgecolor='#0a0e17', alpha=0.85)
ax.axvline(50, color='#64748b', linestyle='--', linewidth=1, label='Coin flip (50%)')
ax.axvline(probs_blend.mean()*100, color='#3b82f6', linewidth=2, label=f'Mean P(UConn) = {probs_blend.mean()*100:.1f}%')
ax.set_xlabel('P(UConn wins) — %')
ax.set_ylabel('Simulation count')
ax.set_title('Championship Predictor — Monte Carlo Distribution (UConn vs Michigan)')
ax.legend()
ax.set_xlim(0, 100)
plt.tight_layout()
hist_path = f'{OUT_DIR}/championship_mc_histogram.png'
plt.savefig(hist_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {hist_path}')

# Save tournament recap as markdown
recap_lines = []
recap_lines.append('# March Madness 2026 — Championship Predictor Results\n')
recap_lines.append(f'Generated from `championship_predictor_colab.ipynb`\n\n')
recap_lines.append('## Model accuracy through Elite Eight\n')
for r, d in STATE['model_accuracy'].items():
    if r == 'cumulative': continue
    recap_lines.append(f"- {r.replace('_',' ').title()}: {d['correct']}/{d['total']} ({d['pct']}%)\n")
c = STATE['model_accuracy']['cumulative']
recap_lines.append(f"- **Cumulative: {c['correct']}/{c['total']} ({c['pct']}%)**\n\n")

recap_lines.append('## Final Four picks\n')
recap_lines.append(f"- Model picks: {STATE['model_final_four_picks']}\n")
recap_lines.append(f"- Correct: {STATE['model_final_four_correct']}/4 (Arizona, Michigan)\n")
recap_lines.append(f"- Original champion pick: {STATE['model_championship_pick']} (eliminated Elite Eight)\n\n")

recap_lines.append('## Championship prediction (UConn #2 vs Michigan #1)\n')
recap_lines.append(f'- Full ensemble: UConn {p_uconn_full*100:.1f}% | Michigan {(1-p_uconn_full)*100:.1f}%\n')
recap_lines.append(f'- Late-round specialist: UConn {p_uconn_late*100:.1f}% | Michigan {(1-p_uconn_late)*100:.1f}%\n')
recap_lines.append(f'- **Blended: UConn {p_uconn*100:.1f}% | Michigan {(1-p_uconn)*100:.1f}%**\n')
recap_lines.append(f'- Predicted winner: **{winner}**\n\n')

recap_lines.append(f'## Monte Carlo ({N_SIMS:,} sims)\n')
recap_lines.append(f'- UConn win share: {uconn_wins/N_SIMS*100:.1f}%\n')
recap_lines.append(f'- Michigan win share: {mich_wins/N_SIMS*100:.1f}%\n')
recap_lines.append(f'- P(UConn) 5th/50th/95th: {p5*100:.1f}% / {p50*100:.1f}% / {p95*100:.1f}%\n\n')

recap_lines.append('## Score projection\n')
recap_lines.append(f'- Projected tempo: {tempo:.1f} possessions\n')
recap_lines.append(f'- UConn {uc:.1f} — Michigan {mi:.1f}\n')
recap_lines.append(f'- {"UConn" if uc>mi else "Michigan"} by {abs(uc-mi):.1f}\n')

recap_path = f'{OUT_DIR}/championship_recap.md'
with open(recap_path, 'w') as f:
    f.writelines(recap_lines)
print(f'Saved: {recap_path}')

print('\nAll files saved to:', OUT_DIR)